# Model Upload

In [ ]:
!python --version

In [ ]:
# Install dependencies
!pip install -q --upgrade numerapi pandas==2.3.1 pyarrow==18.1.0 matplotlib lightgbm scikit-learn scipy cloudpickle==3.1.1

In [ ]:
from numerapi import NumerAPI
import pandas as pd
import json
napi = NumerAPI()

# Pin both dataset and objective for reproducibility. After the v5.3 in-place
# cutover, generic `target` aliases this column, while Ender-20 remains available
# explicitly as `target_ender_20`.
DATA_VERSION = "v5.3"
TARGET_COL = "target_ender_60"
MODEL_ARTIFACT = "example_model_v53_ender60.pkl"

# Download data
napi.download_dataset(f"{DATA_VERSION}/train.parquet")
napi.download_dataset(f"{DATA_VERSION}/features.json")

# Load data
feature_metadata = json.load(open(f"{DATA_VERSION}/features.json"))
features = feature_metadata["feature_sets"]["small"]
# use "medium" or "all" for better performance. Requires more RAM.
# use "quantum" for newest features released in v5.3.
# features = feature_metadata["feature_sets"]["medium"]
# features = feature_metadata["feature_sets"]["all"]
# features = feature_metadata["feature_sets"]["quantum"]
train = pd.read_parquet(
    f"{DATA_VERSION}/train.parquet", columns=["era"] + features + [TARGET_COL]
)

# For better models, join mature validation data and train on all of it.
# This would cause diagnostics to be misleading, so this compact example does not.
# napi.download_dataset(f"{DATA_VERSION}/validation.parquet")
# validation = pd.read_parquet(
#     f"{DATA_VERSION}/validation.parquet",
#     columns=["era", "data_type"] + features + [TARGET_COL],
# )
# validation = validation[validation["data_type"] == "validation"]
# validation = validation.dropna(subset=[TARGET_COL])
# train = pd.concat([train, validation])

# Downsample for speed
train = train[train["era"].isin(train["era"].unique()[::4])]

# Train model
import lightgbm as lgb
model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=5,
    num_leaves=2**5-1,
    colsample_bytree=0.1,
)
model.fit(train[features], train[TARGET_COL])

# Define predict function
def predict(
    live_features: pd.DataFrame,
    live_benchmark_models: pd.DataFrame,
) -> pd.DataFrame:
    live_predictions = model.predict(live_features[features])
    submission = pd.Series(live_predictions, index=live_features.index)
    return submission.to_frame("prediction")

# Pickle predict function with explicit dataset/target provenance in its filename.
import cloudpickle
p = cloudpickle.dumps(predict)
with open(MODEL_ARTIFACT, "wb") as f:
    f.write(p)

# Download file if running in Google Colab
try:
    from google.colab import files
    files.download(MODEL_ARTIFACT)
except ImportError:
    pass
